# Cognite Databricks quickstart (Unity Catalog)

**This is the recommended place to start** for production-style setups: one notebook takes you from install to **UDTFs and Views** registered in **Unity Catalog**, with credentials in **Databricks Secret Manager**.

**You will:**

1. Install `cognite-databricks`.
2. Load a **CDF client** from a TOML file (same pattern as `cognite-pygen`).
3. Create a **Databricks `WorkspaceClient`** and choose a **SQL warehouse** used to run registration SQL.
4. **Generate** Python UDTF modules for your CDF **data model** (via `cognite-pygen-spark`).
5. **Copy** CDF OAuth settings from TOML into a **secret scope** (persisted in Secret Manager).
6. **Register UDTFs** in Unity Catalog, then **register Views** that wrap those UDTFs for simpler SQL.

**Prerequisites:** [Catalog-based prerequisites](../../docs/catalog_based/prerequisites.md) (Unity Catalog, Secret Manager, CDF data model, TOML with `[cognite]` credentials).

**Docs:** [Quickstart (Markdown)](../../docs/catalog_based/quickstart.md) — same flow as this notebook.

## 1. Install

Installs the helper SDK from PyPI. **Restart the Python kernel** if the installer prompts you to.

In [ ]:
# Package: Unity Catalog registration, Secret Manager helpers, and notebook entrypoints.
%pip install --upgrade cognite-databricks

## 2. Imports and CDF client

- **`load_cognite_client_from_toml`**: reads your CDF project/cluster and app registration (used to **fetch the data model** and validate connectivity).
- **TOML path**: must be reachable from this workspace (DBFS or repo path). Values are **re-used** below when writing to Secret Manager.
- **Secret Manager** is where **SQL Warehouses** read credentials at query time — the TOML file is for **provisioning**, not for end users running SQL.

In [ ]:
from cognite.databricks import generate_udtf_notebook
from cognite.client.data_classes.data_modeling.ids import DataModelId
from cognite.pygen import load_cognite_client_from_toml
from databricks.sdk import WorkspaceClient

import toml

# Replace with your workspace path to the CDF config (see docs/catalog_based/prerequisites.md).
toml_file_path = "/Workspace/Users/<your-email>/config/credentials.toml"

# CDF API client — same pattern as pygen notebooks.
client = load_cognite_client_from_toml(toml_file_path)

## 3. Databricks workspace and SQL warehouse

- **`WorkspaceClient`**: uses the notebook’s Databricks auth (pat / OAuth) to call **SQL** and **Secrets** APIs.
- **Warehouses**: registration runs `CREATE FUNCTION` / view DDL via a warehouse — pick the one your org uses for SQL analytics (or any warehouse your principal can use).

In [ ]:
# Authenticates as the current user / service principal in Databricks.
workspace_client = WorkspaceClient()

# List Pro SQL warehouses so you can pick the right ID (name + id).
warehouses = list(workspace_client.warehouses.list())
for w in warehouses:
    print(f"Name: {w.name}, ID: {w.id}")

# Choose explicitly — do not rely on the loop variable after `for`.
if not warehouses:
    raise RuntimeError("No SQL warehouses found. Create one in Databricks SQL.")

# Typical: first warehouse, or filter by name: next(x for x in warehouses if x.name == "My WH")
warehouse = warehouses[0]
print(f"Using warehouse: {warehouse.name} ({warehouse.id})")

## 4. Generate UDTF Python files

- **`DataModelId`**: CDF space, external id, and version — must match the model you want to expose in Databricks.
- **`output_dir`**: generated modules are written here (Databricks workspace path).
- **`catalog` / `schema`**: Unity Catalog location for registered functions and views (`schema` may match your naming convention).
- **`workspace_client`**: required for catalog-based registration and Secret Manager helpers.

In [ ]:
# Target CDF data model (adjust to your model).
data_model_id = DataModelId(space="cdf_cdm", external_id="CogniteCore", version="v1")

# Orchestrates codegen (pygen-spark), file layout, and later registration.
generator = generate_udtf_notebook(
    data_model_id,
    client,
    workspace_client=workspace_client,
    output_dir="/Workspace/Users/<your-email>/udtf_generated",
    catalog="my_catalog",  # Unity Catalog name in your metastore
    schema="CDF_CogniteCore_v1",  # Or None to auto-derive from the data model
    warehouse_id=warehouse.id,
    debug=False,
)

## 5. Persist CDF credentials in Secret Manager

- **Scope name** convention: `cdf_{space}_{external_id_lower}` — matches what generated SQL expects when using `SECRET()`.
- **`set_cdf_credentials`**: creates the scope if needed and writes **project**, **cluster**, **client_id**, **client_secret**, **tenant_id**.
- After this step, **queries do not embed secrets**; they reference `SECRET('scope', 'key')`.

In [ ]:
# Stable scope name derived from the data model (must match registration / SQL examples).
secret_scope = f"cdf_{data_model_id.space}_{data_model_id.external_id.lower()}"

# Re-read TOML to push the same values into Databricks secrets.
toml_content = toml.load(toml_file_path)
cognite_config = toml_content.get("cognite", {})

generator.secret_helper.set_cdf_credentials(
    scope_name=secret_scope,
    project=cognite_config["project"],
    cdf_cluster=cognite_config["cdf_cluster"],
    client_id=cognite_config["client_id"],
    client_secret=cognite_config["client_secret"],
    tenant_id=cognite_config["tenant_id"],
)

## 6. Register UDTFs, then Views

- **`register_udtfs`**: runs Unity Catalog DDL to create **functions** from generated Python files.
- **`register_views`**: creates **views** that call those UDTFs with `SECRET()` bindings so analysts use simple `SELECT * FROM catalog.schema.my_view`.
- **`if_exists`**: `replace` is typical in dev; use `skip` or `error` in stricter environments.
- Set **`debug=True`** for verbose DDL and diagnostics.

In [ ]:
# Step 1: functions in Unity Catalog.
udtf_result = generator.register_udtfs(
    secret_scope=secret_scope,
    if_exists="replace",  # "skip" | "replace" | "error"
    debug=False,
)

# Step 2: views (depend on UDTFs existing).
view_result = generator.register_views(
    secret_scope=secret_scope,
    if_exists="replace",
    debug=False,
)

print("UDTFs:", udtf_result)
print("Views:", view_result)

## 7. Next steps

- **Catalog Explorer**: confirm functions and views under your `catalog.schema`.
- **Querying**: [Querying Views](../../docs/catalog_based/querying.md) — run SQL without pasting credentials.
- **One-shot alternative** (Databricks Runtime **18.1+**): `generator.register_udtfs_and_views(secret_scope=secret_scope, if_exists="replace")`.
- **Session-scoped** (no Unity Catalog): [Session-scoped docs](../../docs/session_scoped/index.md) for ad-hoc development.